# 🎬 Video Object Replacement — CogVideoX Edition

### Принципиальное отличие от SDXL-версии

| | SDXL-версия | **CogVideoX-версия** |
|---|---|---|
| Модель | SDXL Inpainting (image) | **CogVideoX-2B (video)** |
| Обработка | Кадр за кадром независимо | **Чанки по 49 кадров сразу** |
| Временная связность | Оптический поток (пост-обработка) | **Встроена в архитектуру модели** |
| Мерцание | Есть (особенно на быстром движении) | **Минимальное** |
| Скорость | Быстрее | Медленнее |

**CogVideoX** — видео-диффузионная модель от THUDM. Она обрабатывает сразу 49 кадров (~6 секунд) как единое целое через пространственно-временное внимание, что даёт естественную временную связность без дополнительных трюков.

### Инструкция:
1. **Runtime → Change runtime type → T4 GPU** (обязательно!)
2. Запускай ячейки по порядку
3. Загрузи видео, настрой параметры, запусти пайплайн

> ⚠️ CogVideoX-2B весит ~6 GB и скачивается с HuggingFace (~5-10 мин при первом запуске)

## 0. Проверка GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU не найден!\n"
        "Перейди: Runtime → Change runtime type → Hardware accelerator → GPU (T4)"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu_name}")
print(f"   VRAM: {vram_gb:.1f} GB")
print(f"   CUDA: {torch.version.cuda}")

if vram_gb < 13:
    print("\n⚠ VRAM < 13 GB. CogVideoX-2B может не влезть.")
    print("  Попробуй включить CPU offload (параметр ENABLE_CPU_OFFLOAD=True ниже).")

## 1. Установка зависимостей

In [ ]:
import subprocess

def run(cmd):
    print(f"▶ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    lines = (r.stdout or "").strip().splitlines()
    if lines: print("\n".join(lines[-3:]))
    if r.returncode != 0 and r.stderr:
        print(r.stderr[-1000:])

# diffusers >= 0.31 для CogVideoX
run("pip install -q 'diffusers>=0.31.0' transformers accelerate imageio[ffmpeg] opencv-python-headless")

# Grounding DINO + SAM2
run("pip install -q groundingdino-py")
run("pip install -q 'git+https://github.com/facebookresearch/sam2.git'")

print("\n✅ Зависимости установлены!")

## 2. Загрузка весов моделей

In [ ]:
import os

MODELS_DIR = "/content/models"
os.makedirs(MODELS_DIR, exist_ok=True)

SAM2_CHECKPOINT  = f"{MODELS_DIR}/sam2_hiera_large.pt"
GDINO_CHECKPOINT = f"{MODELS_DIR}/groundingdino_swint_ogc.pth"
SAM2_CONFIG      = "configs/sam2.1/sam2.1_hiera_l.yaml"

def download(url, dest):
    if os.path.exists(dest):
        print(f"  ✓ уже есть: {os.path.basename(dest)} ({os.path.getsize(dest)/1e6:.0f} MB)")
        return
    print(f"  ⬇ Качаю {os.path.basename(dest)}...")
    os.system(f'wget -q --show-progress -O "{dest}" "{url}"')
    print(f"  ✓ {os.path.getsize(dest)/1e6:.0f} MB")

print("SAM2 Large:")
download(
    "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt",
    SAM2_CHECKPOINT,
)

print("Grounding DINO:")
download(
    "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth",
    GDINO_CHECKPOINT,
)

# Проверяем SAM2 config в пакете
import sam2 as _s2
_pkg_cfg = os.path.join(os.path.dirname(_s2.__file__), "configs", "sam2.1", "sam2.1_hiera_l.yaml")
print(f"\nSAM2 config: {'✓ найден' if os.path.exists(_pkg_cfg) else '⚠ не найден'}")
print(f"  Путь: {SAM2_CONFIG} (относительный для Hydra)")

print("\n✅ Модели готовы!")

## 3. Код пайплайна

In [ ]:
import cv2
import numpy as np
import torch
import tempfile
import os
import imageio.v2 as imageio
from PIL import Image

DEVICE = "cuda"
DTYPE  = torch.bfloat16  # CogVideoX рекомендует bfloat16

# CogVideoX нативное разрешение: 480×720 (portrait) или 720×480 (landscape).
# Мы используем квадратный кроп 480×480 — работает на T4.
COGVIDEO_H = 480
COGVIDEO_W = 480

print(f"Device: {DEVICE} | dtype: {DTYPE}")
print(f"Разрешение для CogVideoX: {COGVIDEO_W}×{COGVIDEO_H}")

In [ ]:
# ── Извлечение кадров ──────────────────────────────────────────────────

def extract_frames(video_path: str, target_h: int, target_w: int, max_frames=None):
    cap = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS) or 24.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (target_w, target_h))
        frames.append(frame)
        if max_frames and len(frames) >= max_frames:
            break
    cap.release()
    print(f"  Извлечено {len(frames)}/{total} кадров @ {fps:.1f} FPS")
    return frames, fps


def scan_video_for_object(video_path, gdino_model, text_prompt,
                           box_threshold, text_threshold, scan_frames=30):
    """Сканирует ВСЁ видео без загрузки всех кадров в RAM."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 24.0
    cap.release()

    step = max(1, total // scan_frames)
    probe_indices = [int(i * total / scan_frames) for i in range(scan_frames)]
    print(f"  Видео: {total} кадров @ {fps:.1f} FPS | проверяю {len(probe_indices)} кадров")

    detections = []
    cap = cv2.VideoCapture(video_path)
    for fi in probe_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret:
            continue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_rgb = cv2.resize(frame_rgb, (COGVIDEO_W, COGVIDEO_H))
        result = detect_bbox(gdino_model, frame_rgb, text_prompt,
                              box_threshold, text_threshold)
        if result is not None:
            bbox, phrase, conf = result if len(result) == 3 else (*result, 0.5)
            detections.append((conf, fi, bbox, phrase, frame_rgb))
            print(f"  Кадр {fi:4d}/{total}: '{phrase}'  conf={conf:.3f} ✓")
        else:
            print(f"  Кадр {fi:4d}/{total}: не найдено")
    cap.release()
    return detections


# ── Grounding DINO ─────────────────────────────────────────────────────

def load_grounding_dino(checkpoint_path):
    import groundingdino
    from groundingdino.util.inference import load_model
    config = os.path.join(os.path.dirname(groundingdino.__file__),
                          "config", "GroundingDINO_SwinT_OGC.py")
    model = load_model(config, checkpoint_path, device=DEVICE)
    print("  ✓ Grounding DINO загружен")
    return model


def detect_bbox(model, image_np, text_prompt, box_threshold=0.35, text_threshold=0.25):
    import torchvision.transforms as T
    from groundingdino.util.inference import predict
    transform = T.Compose([
        T.Resize((800, 800)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img_t = transform(Image.fromarray(image_np))
    h, w = image_np.shape[:2]
    boxes, logits, phrases = predict(
        model=model, image=img_t, caption=text_prompt,
        box_threshold=box_threshold, text_threshold=text_threshold, device=DEVICE,
    )
    if len(boxes) == 0:
        return None
    best = int(logits.argmax())
    cx, cy, bw, bh = boxes[best].tolist()
    x1 = max(0, int((cx - bw/2) * w))
    y1 = max(0, int((cy - bh/2) * h))
    x2 = min(w, int((cx + bw/2) * w))
    y2 = min(h, int((cy + bh/2) * h))
    return (x1, y1, x2, y2), phrases[best], float(logits[best])


# ── SAM2 ───────────────────────────────────────────────────────────────

def build_sam2_image_predictor(checkpoint, config):
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    model = build_sam2(config, checkpoint, device=DEVICE)
    print("  ✓ SAM2 Image Predictor загружен")
    return SAM2ImagePredictor(model)


def get_mask_from_bbox(predictor, image_np, bbox):
    x1, y1, x2, y2 = bbox
    predictor.set_image(image_np)
    masks, _, _ = predictor.predict(
        box=np.array([x1, y1, x2, y2]), multimask_output=False,
    )
    return masks[0].astype(np.uint8) * 255


def build_sam2_video_predictor(checkpoint, config):
    from sam2.build_sam import build_sam2_video_predictor
    pred = build_sam2_video_predictor(config, checkpoint, device=DEVICE)
    print("  ✓ SAM2 Video Predictor загружен")
    return pred


def _write_frames(frames, directory):
    for i, f in enumerate(frames):
        bgr = cv2.cvtColor(f, cv2.COLOR_RGB2BGR)
        cv2.imwrite(os.path.join(directory, f"{i:05d}.jpg"), bgr,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])


def track_masks_video(predictor, frames, initial_mask, seed_idx=0):
    n = len(frames)
    with tempfile.TemporaryDirectory() as tmpdir:
        _write_frames(frames, tmpdir)
        with torch.inference_mode():
            state = predictor.init_state(tmpdir)
            predictor.reset_state(state)
            predictor.add_new_mask(
                inference_state=state, frame_idx=seed_idx,
                obj_id=1, mask=initial_mask > 128,
            )
            tracked = [None] * n
            for out_idx, obj_ids, logits in predictor.propagate_in_video(state):
                if 1 not in obj_ids:
                    continue
                ch = list(obj_ids).index(1)
                binary = (logits[ch] > 0.0).squeeze(0).cpu().numpy()
                tracked[out_idx] = binary.astype(np.uint8) * 255
    empty = np.zeros(initial_mask.shape, dtype=np.uint8)
    return [m if m is not None else empty.copy() for m in tracked]


def fill_mask_gaps(masks, max_gap=2):
    n = len(masks)
    filled = [m.copy() for m in masks]
    i = 0
    while i < n:
        if filled[i].any():
            i += 1
            continue
        j = i
        while j < n and not filled[j].any():
            j += 1
        if (j - i) <= max_gap and i > 0 and j < n:
            for k in range(i, j):
                t = (k - i + 1) / (j - i + 1)
                blended = (1-t)*filled[i-1].astype(np.float32) + t*filled[j].astype(np.float32)
                filled[k] = (blended > 127).astype(np.uint8) * 255
        i = j
    return filled


print("✅ Все функции готовы!")

In [ ]:
# ── CogVideoX Video-to-Video Pipeline ─────────────────────────────────

def load_cogvideo_pipeline(model_id: str, enable_cpu_offload: bool = False):
    from diffusers import CogVideoXVideoToVideoPipeline

    print(f"  Загружаю {model_id} (может занять несколько минут при первом запуске)...")
    pipe = CogVideoXVideoToVideoPipeline.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
    )

    if enable_cpu_offload:
        pipe.enable_model_cpu_offload()
        print("  ✓ CPU offload включён (медленнее, но экономит VRAM)")
    else:
        pipe = pipe.to(DEVICE)

    pipe.vae.enable_tiling()        # разбивает VAE на тайлы — экономит ~2 GB VRAM
    pipe.vae.enable_slicing()       # обрабатывает кадры батчами — экономит ещё ~1 GB
    pipe.set_progress_bar_config(disable=True)
    print("  ✓ CogVideoX pipeline загружен")
    return pipe


# ── Подготовка кадров для CogVideoX ───────────────────────────────────

def prefill_masked_frames(frames, masks):
    """
    Заполняем область маски размытым содержимым окружения.
    Это даёт CogVideoX корректный фон без объекта для ориентации.
    """
    result = []
    for frame, mask in zip(frames, masks):
        if not mask.any():
            result.append(frame.copy())
            continue
        # Размытие большим ядром — убирает оригинальный объект
        blurred = cv2.GaussianBlur(frame, (99, 99), 0)
        mask_f = (mask[..., None] / 255.0).astype(np.float32)
        filled = frame.astype(np.float32) * (1 - mask_f) + blurred.astype(np.float32) * mask_f
        result.append(filled.astype(np.uint8))
    return result


def composite_frames(original_frames, generated_frames, masks, feather_px=15):
    """
    Compositing: берём пиксели из generated только внутри маски,
    снаружи — оригинал. Граница маски размывается для плавного перехода.
    """
    composed = []
    for orig, gen, mask in zip(original_frames, generated_frames, masks):
        if not mask.any():
            composed.append(orig.copy())
            continue
        # Размываем маску для мягких краёв
        k = feather_px * 2 + 1
        mask_f = cv2.GaussianBlur(mask.astype(np.float32) / 255.0, (k, k), feather_px // 2)
        mask_f = mask_f[..., None]  # [H, W, 1]
        result = gen.astype(np.float32) * mask_f + orig.astype(np.float32) * (1 - mask_f)
        composed.append(result.clip(0, 255).astype(np.uint8))
    return composed


# ── Chunk-based обработка ──────────────────────────────────────────────

def frames_to_pil(frames):
    return [Image.fromarray(f) for f in frames]


def process_chunk(pipe, frames_chunk, masks_chunk, prompt, negative_prompt,
                  strength, num_steps, seed, target_h, target_w):
    """
    Обрабатывает один чанк через CogVideoX video-to-video.
    Возвращает numpy-кадры после compositing.
    """
    # CogVideoX требует нечётное число кадров: 9, 17, 25, 33, 41, 49
    n = len(frames_chunk)
    target_n = max(9, n if n % 8 == 1 else (n // 8) * 8 + 1)

    # Дополняем до нужного размера если нужно
    padded_frames = frames_chunk + [frames_chunk[-1]] * (target_n - n)
    padded_masks  = masks_chunk  + [masks_chunk[-1]]  * (target_n - n)

    # Заполняем маску
    prefilled = prefill_masked_frames(padded_frames, padded_masks)

    generator = torch.Generator(DEVICE).manual_seed(seed)

    with torch.no_grad():
        result = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            video=frames_to_pil(prefilled),
            strength=strength,
            num_inference_steps=num_steps,
            guidance_scale=6.0,
            generator=generator,
            height=target_h,
            width=target_w,
            use_dynamic_cfg=True,
        )

    # Извлекаем кадры из результата
    gen_frames_raw = result.frames[0]  # list of PIL или tensor
    gen_frames = []
    for f in gen_frames_raw:
        arr = np.array(f) if isinstance(f, Image.Image) else (f.numpy() * 255).astype(np.uint8)
        if arr.shape[:2] != (target_h, target_w):
            arr = cv2.resize(arr, (target_w, target_h))
        gen_frames.append(arr)

    # Compositing — вставляем генерацию только в область маски
    composed = composite_frames(padded_frames, gen_frames, padded_masks)

    # Возвращаем только реальные (не паддинговые) кадры
    return composed[:n]


def process_video_cogvideo(pipe, frames, masks, prompt, negative_prompt,
                            strength, num_steps, seed,
                            chunk_size=49, overlap=8,
                            target_h=480, target_w=480):
    """
    Обрабатывает полное видео чанками по chunk_size кадров с overlap перекрытием.
    Перекрывающиеся кадры плавно смешиваются для бесшовных переходов.
    """
    n = len(frames)
    # chunk_size должен давать нечётное число: 49 = 6*8+1 ✓
    # Убеждаемся что chunk_size нечётный
    if chunk_size % 2 == 0:
        chunk_size += 1

    step = chunk_size - overlap
    starts = list(range(0, n, step))

    # Буфер результатов — каждая позиция может писаться несколько раз (overlap)
    result_buffer = [None] * n      # финальный кадр
    weight_buffer = [0.0] * n       # вес для усреднения в overlap-зонах

    for ci, start in enumerate(starts):
        end = min(start + chunk_size, n)
        chunk_frames = frames[start:end]
        chunk_masks  = masks[start:end]

        # Пропускаем чанки без маски — объект не виден, оставляем оригинал
        if not any(m.any() for m in chunk_masks):
            for i in range(start, end):
                if result_buffer[i] is None:
                    result_buffer[i] = frames[i].copy()
                    weight_buffer[i] = 1.0
            print(f"  Чанк {ci+1}/{len(starts)}: кадры {start}-{end-1} — объект не виден, пропускаю")
            continue

        print(f"  Чанк {ci+1}/{len(starts)}: кадры {start}-{end-1} → CogVideoX...")

        chunk_result = process_chunk(
            pipe, chunk_frames, chunk_masks,
            prompt, negative_prompt, strength, num_steps, seed + ci,
            target_h, target_w,
        )

        # Записываем в буфер с линейными весами для overlap-зон
        for local_i, global_i in enumerate(range(start, end)):
            if global_i >= n:
                break
            # Вес: 1.0 в центре чанка, убывает к краям (для плавного сшивания)
            chunk_len = end - start
            edge_dist = min(local_i, chunk_len - 1 - local_i)
            w = min(1.0, (edge_dist + 1) / max(overlap, 1))

            if result_buffer[global_i] is None:
                result_buffer[global_i] = chunk_result[local_i].astype(np.float32)
                weight_buffer[global_i] = w
            else:
                # Взвешенное усреднение в overlap-зоне
                result_buffer[global_i] = (
                    result_buffer[global_i] * weight_buffer[global_i] +
                    chunk_result[local_i].astype(np.float32) * w
                ) / (weight_buffer[global_i] + w)
                weight_buffer[global_i] += w

        torch.cuda.empty_cache()

    # Финализируем: приводим к uint8
    final = []
    for i, f in enumerate(result_buffer):
        if f is None:
            final.append(frames[i].copy())
        else:
            final.append(f.clip(0, 255).astype(np.uint8))
    return final


print("✅ CogVideoX функции готовы!")

## 4. Загрузка видео

In [ ]:
# ── Вариант A: загрузить файл с компьютера ────────────────────────────
from google.colab import files
uploaded = files.upload()
INPUT_VIDEO = list(uploaded.keys())[0]
print(f"Загружено: {INPUT_VIDEO}  ({os.path.getsize(INPUT_VIDEO)/1e6:.1f} MB)")

In [ ]:
# ── Вариант B: Google Drive ───────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# INPUT_VIDEO = "/content/drive/MyDrive/my_video.mp4"

## 5. Параметры

In [ ]:
# ┌─────────────────────────────────────────────────────────────────────┐
# │                     НАСТРОЙКИ                                       │
# └─────────────────────────────────────────────────────────────────────┘

# Что найти (Grounding DINO)
DETECT_PROMPT  = "clock watch"
BOX_THRESHOLD  = 0.35
TEXT_THRESHOLD = 0.25
SCAN_FRAMES    = 30  # сканируем всё видео целиком

# Что сгенерировать (CogVideoX prompt)
# Совет: описывай объект В ДВИЖЕНИИ — CogVideoX понимает динамику
PROMPT = (
    "a ceramic mug with space galaxy design, sitting on a surface, "
    "photorealistic, cinematic lighting, 4k"
)
NEGATIVE_PROMPT = "distorted, blurry, extra objects, low quality, watermark"

# Модель CogVideoX
# "THUDM/CogVideoX-2b" — меньше (6 GB), быстрее, подходит для T4
# "THUDM/CogVideoX-5b" — качественнее, нужен A100 (40 GB VRAM)
COGVIDEO_MODEL = "THUDM/CogVideoX-2b"

# Параметры генерации
STRENGTH       = 0.8   # насколько сильно менять кадры (0.5=мало, 0.95=сильно)
NUM_STEPS      = 50    # шаги диффузии (25=быстро, 50=качественно)
SEED           = 42

# Параметры чанков
# CogVideoX обрабатывает строго 2k+1 кадров: 9, 17, 25, 33, 41, 49
CHUNK_SIZE     = 49    # кадров за один проход CogVideoX
CHUNK_OVERLAP  = 8     # кадров перекрытия между чанками (для плавного сшивания)

# ENABLE_CPU_OFFLOAD: True если мало VRAM (T4 с другими моделями в памяти)
ENABLE_CPU_OFFLOAD = False

# DEBUG: None = обработать всё видео. Число = только первые N кадров.
DEBUG_MAX_FRAMES = None

OUTPUT_VIDEO = "/content/output_cogvideo.mp4"

print("Параметры:")
print(f"  Что найти:   '{DETECT_PROMPT}'")
print(f"  Промпт:      '{PROMPT[:60]}...'")
print(f"  Модель:      {COGVIDEO_MODEL}")
print(f"  Strength:    {STRENGTH}  |  Steps: {NUM_STEPS}")
print(f"  Chunk:       {CHUNK_SIZE} кадров, overlap={CHUNK_OVERLAP}")
print(f"  Обработка:   {'ВСЁ видео' if not DEBUG_MAX_FRAMES else f'{DEBUG_MAX_FRAMES} кадров (DEBUG)'}")

## 6. Запуск пайплайна

In [ ]:
# ── Шаг 1: Поиск объекта по всему видео ───────────────────────────────
print("=" * 60)
print("Шаг 1: Поиск объекта (Grounding DINO — сканирую всё видео)")
print("=" * 60)

gdino = load_grounding_dino(GDINO_CHECKPOINT)
detections = scan_video_for_object(
    INPUT_VIDEO, gdino, DETECT_PROMPT, BOX_THRESHOLD, TEXT_THRESHOLD, SCAN_FRAMES
)

if not detections:
    raise RuntimeError(
        f"Объект '{DETECT_PROMPT}' не найден.\n"
        f"Попробуй: снизь BOX_THRESHOLD до 0.20, или измени DETECT_PROMPT"
    )

detections.sort(reverse=True)
best_conf, seed_frame_idx, found_bbox, found_phrase, seed_frame_rgb = detections[0]
print(f"\n  ★ Лучший: '{found_phrase}' на кадре {seed_frame_idx}  conf={best_conf:.3f}")
if best_conf < 0.40:
    print(f"  ⚠ Уверенность невысокая — подними BOX_THRESHOLD если нашёлся не тот объект")

In [ ]:
# ── Шаг 2: Извлечение всех кадров ─────────────────────────────────────
print("=" * 60)
print("Шаг 2: Извлечение кадров")
print("=" * 60)

frames, fps = extract_frames(
    INPUT_VIDEO, COGVIDEO_H, COGVIDEO_W, max_frames=DEBUG_MAX_FRAMES
)
local_seed_idx = min(seed_frame_idx, len(frames) - 1)
print(f"  Seed-кадр: {local_seed_idx}")

# ── Шаг 3: Начальная маска (SAM2 Image) ───────────────────────────────
print("\n" + "=" * 60)
print("Шаг 3: Создание начальной маски (SAM2)")
print("=" * 60)

sam_img = build_sam2_image_predictor(SAM2_CHECKPOINT, SAM2_CONFIG)
initial_mask = get_mask_from_bbox(sam_img, frames[local_seed_idx], found_bbox)
del sam_img
torch.cuda.empty_cache()
print(f"  ✓ Пикселей в маске: {initial_mask.sum() // 255}")

# Превью
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(frames[local_seed_idx])
x1, y1, x2, y2 = found_bbox
axes[0].add_patch(plt.Rectangle((x1,y1), x2-x1, y2-y1,
                                  fill=False, edgecolor='lime', linewidth=3))
axes[0].set_title(f"'{found_phrase}'  conf={best_conf:.3f}", fontsize=13)
axes[0].axis('off')
overlay = frames[local_seed_idx].copy()
overlay[initial_mask > 128] = [255, 80, 80]
axes[1].imshow((0.55*frames[local_seed_idx] + 0.45*overlay).astype(np.uint8))
axes[1].set_title("Маска SAM2 (красное = будет заменено)", fontsize=13)
axes[1].axis('off')
plt.tight_layout()
plt.show()
print("\n  ↑ Проверь: красная зона должна покрывать нужный объект.")

In [ ]:
# ── Шаг 4: Отслеживание маски (SAM2 Video) ────────────────────────────
print("=" * 60)
print("Шаг 4: Отслеживание объекта (SAM2 Video Predictor)")
print("=" * 60)

sam_vid = build_sam2_video_predictor(SAM2_CHECKPOINT, SAM2_CONFIG)
tracked_masks = track_masks_video(sam_vid, frames, initial_mask, local_seed_idx)
del sam_vid
torch.cuda.empty_cache()

tracked_masks = fill_mask_gaps(tracked_masks, max_gap=2)
visible = sum(m.any() for m in tracked_masks)
print(f"  ✓ Объект виден в {visible}/{len(frames)} кадрах")

# ── Шаг 5: Загрузка CogVideoX ─────────────────────────────────────────
print("\n" + "=" * 60)
print("Шаг 5: Загрузка CogVideoX")
print("=" * 60)

del gdino
torch.cuda.empty_cache()

cogvideo_pipe = load_cogvideo_pipeline(COGVIDEO_MODEL, ENABLE_CPU_OFFLOAD)

vram_free = torch.cuda.mem_get_info()[0] / 1e9
print(f"  Свободно VRAM: {vram_free:.1f} GB")

In [ ]:
# ── Шаг 6: Обработка видео чанками (CogVideoX) ────────────────────────
import time

print("=" * 60)
print("Шаг 6: Замена объекта (CogVideoX video-to-video)")
print("=" * 60)
print(f"  Кадров: {len(frames)}  |  Чанк: {CHUNK_SIZE}  |  Overlap: {CHUNK_OVERLAP}")
step = CHUNK_SIZE - CHUNK_OVERLAP
n_chunks = max(1, (len(frames) + step - 1) // step)
est_min = n_chunks * NUM_STEPS * 0.6 / 60  # грубая оценка
print(f"  Чанков: {n_chunks}  |  Оценка времени: ~{est_min:.0f}-{est_min*1.5:.0f} мин на T4")
print()

t0 = time.time()
processed = process_video_cogvideo(
    cogvideo_pipe, frames, tracked_masks,
    PROMPT, NEGATIVE_PROMPT,
    strength=STRENGTH,
    num_steps=NUM_STEPS,
    seed=SEED,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP,
    target_h=COGVIDEO_H,
    target_w=COGVIDEO_W,
)
elapsed = time.time() - t0
print(f"\n✅ Обработка завершена за {elapsed/60:.1f} мин")

In [ ]:
# ── Шаг 7: Сборка видео и превью ──────────────────────────────────────
print("Собираю видео...")
imageio.mimsave(OUTPUT_VIDEO, processed, fps=fps)
print(f"✅ Сохранено: {OUTPUT_VIDEO}  ({os.path.getsize(OUTPUT_VIDEO)/1e6:.1f} MB)")

# Сравнение до/после на нескольких кадрах
preview_idxs = [i for i in range(0, len(frames), max(1, len(frames)//4))][:4]
fig, axes = plt.subplots(len(preview_idxs), 2, figsize=(12, 5*len(preview_idxs)))
if len(preview_idxs) == 1:
    axes = [axes]
for row, fi in enumerate(preview_idxs):
    axes[row][0].imshow(frames[fi])
    axes[row][0].set_title(f"Кадр {fi} — ДО", fontsize=13)
    axes[row][0].axis('off')
    axes[row][1].imshow(processed[fi])
    axes[row][1].set_title(f"Кадр {fi} — ПОСЛЕ (CogVideoX)", fontsize=13)
    axes[row][1].axis('off')
plt.suptitle("До / После", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Шаг 8: Скачать результат ──────────────────────────────────────────
from google.colab import files
files.download(OUTPUT_VIDEO)
print("✅ Файл отправлен в загрузки!")

---

## Советы по настройке

### Параметр `STRENGTH`
- `0.5–0.6` — лёгкое изменение, объект будет похож на оригинал
- `0.7–0.8` — стандарт, хороший баланс
- `0.85–0.95` — агрессивное изменение, максимальное следование промпту

### Параметр `CHUNK_SIZE`
CogVideoX принимает строго `2k+1` кадров: **9, 17, 25, 33, 41, 49**
- `49` — максимальный (~6 сек), лучшая временная связность
- `25` — меньше памяти, быстрее

### Промпт для CogVideoX
В отличие от SDXL, CogVideoX понимает **динамику**. Описывай поведение:
- ❌ `"ceramic mug"` — слишком коротко
- ✅ `"a ceramic mug with galaxy design resting on a wooden table, soft studio lighting, photorealistic, 4k"`

### Если мало VRAM (OOM)
1. Поставь `ENABLE_CPU_OFFLOAD = True`
2. Уменьши `CHUNK_SIZE` до `25`
3. Поставь `DEBUG_MAX_FRAMES = 49` для быстрого теста

### Объект не найден
- Снизь `BOX_THRESHOLD` до `0.20`
- Попробуй простые слова: `mug`, `cup`, `watch`, `bottle`, `clock`